In [67]:
folder_names = ["access_control", "arithmetic", "denial_of_service", "front_running", "reentrancy", "time_manipulation", "unchecked_low_level_calls"]

detection_labels = {"access_control": "[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]", 
                    "arithmetic": "[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]", 
                    "denial_of_service": "[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]", 
                    "front_running": "[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]", 
                    "reentrancy": "[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]", 
                    "time_manipulation": "[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]", 
                    "unchecked_low_level_calls": "[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]"}

In [75]:
import os
import pandas as pd

# Output storage
data = []

# Base directory (update if needed)
base_dir = "."

for folder in folder_names:
    folder_path = os.path.join(base_dir, folder)
    if not os.path.isdir(folder_path):
        print(f"Folder not found: {folder_path}")
        continue

    # Get .sol files and sort
    sol_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".sol")])

    for file_name in sol_files:
        file_path = os.path.join(folder_path, file_name)
        file_index = file_name.split("_")[1].split(".")[0]
        csv_name = "BugLog_" + file_index + ".csv"
        csv_path = os.path.join(folder_path, csv_name)

        # Default value if CSV not found
        num_injections = 0

        # Try reading the corresponding CSV
        if os.path.isfile(csv_path):
            try:
                df_injection = pd.read_csv(csv_path)
                num_injections = len(df_injection)
            except Exception as e:
                print(f"Error reading {csv_path}: {e}")

        # Read and clean the Solidity file
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                cleaned_lines = [line.rstrip() for line in lines if line.strip() != ""]
                cleaned_code = "\n".join(cleaned_lines)

                assert "\n\n" not in cleaned_code, f"Double newlines still exist in {file_name}"
                data.append({
                    "filename": file_name,
                    "source_code": cleaned_code,
                    "type": folder,
                    "num_injections": num_injections,
                    "detection_label": detection_labels[folder]
                })
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

# Convert to DataFrame and save
df = pd.DataFrame(data)
output_csv = "vulnerability_dataset.csv"
df.to_csv(output_csv, index=False, encoding='utf-8')
print(f"Saved dataset with injection counts to {output_csv}")

Saved dataset with injection counts to vulnerability_dataset.csv


In [70]:
df = pd.read_csv("vulnerability_dataset.csv")

df_localize = pd.DataFrame()
df_localize['item_index'] = [i for i in range(len(df))]
df_localize['source_code'] = df['source_code']
df_localize['vuln_lines'] = "[]"

df_localize.to_csv("solidify_localize.csv", index=False, encoding='utf-8')

In [72]:
df = pd.read_csv("vulnerability_dataset.csv")

df_localize = pd.DataFrame()
df_localize['item_index'] = [i for i in range(len(df))]
df_localize['source_code'] = df['source_code']
df_localize['vuln_types'] = df['detection_label']

df_localize.to_csv("solidify_detect.csv", index=False, encoding='utf-8')